# UrbanRM7 — getting started

UrbanRM7 contains **2400 ray-traced path-loss maps at 7 GHz** for 150 urban tiles of 256 m × 256 m (1 m pixels) in 15 cities,
computed with Sionna RT (specular reflection, wedge and rooftop diffraction, diffuse scattering, up to five interactions,
3×10⁹ rays per transmitter). Every map comes with the 17-channel geometric feature stack used in the paper
*To Deep Learn or Not: Accuracy, Data and Cost of Radio Map Prediction in the Upper Mid-Band*.

This notebook shows how to

1. download the dataset (or point to a local copy),
2. read tiles, path-loss maps and features,
3. plot maps and dataset statistics,
4. train LightGBM on a **selectable subset of features** and evaluate it on the unseen test cities,
5. train a small U-Net on the same features and compare,
6. look at the errors per city and per LoS/NLoS condition.

It runs on Google Colab or locally. The U-Net part uses a GPU when one is available.

## 0. Settings

In [ ]:
# ---- where the data live ----
DATA_DIR = "UrbanRM7"          # folder that contains metadata/, data/, ... (created by the download below)
DOWNLOAD = True                # set False if the dataset is already in DATA_DIR
RELEASE_URL = "https://github.com/sajjadhussa1n/radiomap-ml-vs-dl/releases/download/v1.0"
PARTS = ["metadata_results", "test", "val", "train"]   # add "scenes" for the Sionna meshes

# ---- experiment size (reduce for a quick run) ----
N_TRAIN_TILES = 30             # training tiles used here (100 available, 16 maps each)
PIX_PER_MAP = 1500             # pixels sampled per training map for LightGBM (as in the paper)
EVAL_PIX_PER_MAP = 2048        # pixels evaluated per test map (as in the paper)
UNET_EPOCHS = 15               # U-Net training epochs
SEED = 0
FLOOR_DB = 160.0               # label policy of the paper: path loss clipped at 160 dB, unreached pixels = 160 dB

## 1. Download

In [ ]:
import os, zipfile, hashlib, urllib.request, urllib.error
from pathlib import Path

def _fetch(url, dst):
    try:
        with urllib.request.urlopen(url) as r, open(dst, "wb") as fh:
            total, done = int(r.headers.get("Content-Length", 0)), 0
            while True:
                b = r.read(1 << 22)
                if not b:
                    break
                fh.write(b); done += len(b)
                print(f"\r  {dst.name}: {done / 2**20:,.0f} / {total / 2**20:,.0f} MB", end="")
        print()
        return True
    except urllib.error.HTTPError as e:
        if dst.exists():
            dst.unlink()
        if e.code == 404:
            return False
        raise

def download(parts, data_dir=DATA_DIR, url=RELEASE_URL):
    root = Path(data_dir).parent if Path(data_dir).name == "UrbanRM7" else Path(data_dir)
    root.mkdir(parents=True, exist_ok=True)
    for p in parts:
        names = [f"UrbanRM7_{p}.zip"]
        if not _fetch(f"{url}/{names[0]}", root / names[0]):      # large parts may be split into _part1, _part2, ...
            names, k = [], 1
            while _fetch(f"{url}/UrbanRM7_{p}_part{k}.zip", root / f"UrbanRM7_{p}_part{k}.zip"):
                names.append(f"UrbanRM7_{p}_part{k}.zip"); k += 1
        for n in names:
            with zipfile.ZipFile(root / n) as z:
                z.extractall(root)
            (root / n).unlink()

if DOWNLOAD and not (Path(DATA_DIR) / "metadata" / "tiles.csv").exists():
    download(PARTS)
print(sorted(os.listdir(DATA_DIR)))

## 2. Dataset structure

```
UrbanRM7/
  metadata/tiles.csv      one row per tile: city, split, centre (lat/lon), building cover, heights
  metadata/maps.csv       one row per map: tile, transmitter index and position, floor share, RT time
  metadata/channels.csv   the 17 feature channels (name, unit, description)
  metadata/config.json    ray-tracing settings, materials, grid convention, label policy
  data/{train,val,test}/<scene_id>.npz   one file per tile (16 maps), see below
  scenes/<scene_id>/      Sionna scene (scene.xml + PLY meshes) to re-run or extend the ray tracing
  checks/                 LoS-only runs and a second Monte-Carlo seed (label-noise check)
  results/                all result files of the paper
```

Each tile file holds

| key | shape | content |
|---|---|---|
| `pathloss_db` | 16×256×256 float32 | raw path loss, NaN where no path reaches |
| `reached` | 16×256×256 bool | outdoor pixel reached by at least one path |
| `outdoor` | 256×256 bool | pixel outside buildings |
| `height_m`, `height_fine_m` | 256×256, 512×512 | building height raster (1 m and 0.5 m) |
| `features_scene` | 7×256×256 float16 | transmitter-independent channels |
| `features_tx` | 16×10×256×256 float16 | transmitter-dependent channels |
| `tx_xyz_m` | 16×3 | transmitter position (tile coordinates, metres) |
| `rt_seconds` | 16 | ray-tracing time per map on a T4 GPU |

Row *i* of a map is the *y* coordinate and column *j* the *x* coordinate, both increasing, with the origin at the tile centre.

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path(DATA_DIR)
tiles = pd.read_csv(ROOT / "metadata" / "tiles.csv")
maps = pd.read_csv(ROOT / "metadata" / "maps.csv")
channels = pd.read_csv(ROOT / "metadata" / "channels.csv")
cfg = json.load(open(ROOT / "metadata" / "config.json"))
CH = channels["name"].tolist()
print(tiles.groupby("split").agg(tiles=("scene_id", "count"), cities=("city", lambda c: ", ".join(sorted(set(c))))).to_string())
print(f"\n{len(maps)} maps, frequency {cfg['frequency_ghz']} GHz, pixel {cfg['pixel_m']} m, Rx height {cfg['rx_height_m']} m")
channels

## 3. Reading a tile

In [ ]:
def load_tile(scene_id):
    split = tiles.loc[tiles.scene_id == scene_id, "split"].item()
    with np.load(ROOT / "data" / split / f"{scene_id}.npz") as d:
        return {k: d[k] for k in d.files}

def label(pl, floor=FLOOR_DB):
    # Label policy of the paper: unreached pixels and path loss above the floor are set to the floor.
    return np.minimum(np.nan_to_num(pl, nan=floor), floor)

def feature_stack(tile, k, names=CH):
    # Feature image [len(names), 256, 256] of transmitter k, channels selected by name.
    full = np.concatenate([tile["features_scene"], tile["features_tx"][k]], 0).astype(np.float32)
    return full[[CH.index(n) for n in names]]

sid = tiles[tiles.split == "test"].scene_id.iloc[0]
T = load_tile(sid)
{k: (v.shape, v.dtype) for k, v in T.items()}

## 4. Example plots

In [ ]:
os.makedirs("figures", exist_ok=True)
h = cfg["tile_m"] / 2
ext = [-h, h, -h, h]
los_frac = [(T["features_tx"][j, CH.index("los") - 7][T["outdoor"]] > 0.5).mean() for j in range(16)]
k = int(np.argmax(los_frac))     # transmitter with the largest LoS area (some rooftop sites see no street at all)
print(f"LoS share of the outdoor pixels per transmitter: {np.round(los_frac, 2)}; showing Tx {k}")
y = np.where(T["outdoor"], label(T["pathloss_db"][k]), np.nan)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
im = ax[0].imshow(T["height_m"], origin="lower", extent=ext, cmap="Greys"); fig.colorbar(im, ax=ax[0], label="height (m)")
im = ax[1].imshow(y, origin="lower", extent=ext, cmap="viridis_r"); fig.colorbar(im, ax=ax[1], label="path loss (dB)")
for a in ax[:2]:
    a.plot(*T["tx_xyz_m"][k, :2], "r*", ms=12)
ax[2].hist(y[np.isfinite(y)], bins=80, color="tab:blue"); ax[2].set_xlabel("path loss (dB)"); ax[2].set_ylabel("pixels")
ax[0].set_title(f"{sid}: buildings"); ax[1].set_title(f"Tx {k}: ray-traced path loss"); ax[2].set_title("histogram")
fig.tight_layout(); fig.savefig("figures/tile_example.png", dpi=150); plt.show()

In [ ]:
show = ["bfrac25", "dist_bldg", "logd3", "los", "ncross", "nu"]
X = feature_stack(T, k, show)
fig, ax = plt.subplots(1, len(show), figsize=(3 * len(show), 3))
for a, n, img in zip(ax, show, X):
    im = a.imshow(np.where(T["outdoor"], img, np.nan), origin="lower", extent=ext, cmap="magma")
    a.set_title(n); a.set_xticks([]); a.set_yticks([]); fig.colorbar(im, ax=a, fraction=0.046)
fig.tight_layout(); fig.savefig("figures/features_example.png", dpi=150); plt.show()

In [ ]:
# Path loss against 3-D distance, LoS and NLoS (one map per test tile)
rng = np.random.default_rng(SEED)
pts = []
for s in tiles[tiles.split == "test"].scene_id:
    t = load_tile(s)
    out = np.flatnonzero(t["outdoor"].ravel())
    idx = rng.choice(out, min(3000, len(out)), replace=False)
    f = feature_stack(t, 0, ["logd3", "los"]).reshape(2, -1)[:, idx]
    pts.append(np.c_[f.T, label(t["pathloss_db"][0]).ravel()[idx]])
P = np.concatenate(pts)
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for v, c, n in [(1, "tab:orange", "LoS"), (0, "tab:blue", "NLoS")]:
    s = P[:, 1] == v
    ax.scatter(10 ** P[s, 0], P[s, 2], s=2, alpha=0.3, c=c, label=n)
d = np.logspace(0.5, 2.6, 50)
ax.plot(d, 20 * np.log10(d) + 20 * np.log10(cfg["frequency_ghz"] * 1e9) - 147.55, "k--", label="free space")
ax.set_xscale("log"); ax.set_xlabel("3-D distance (m)"); ax.set_ylabel("path loss (dB)"); ax.legend(markerscale=5)
fig.tight_layout(); fig.savefig("figures/pathloss_vs_distance.png", dpi=150); plt.show()

In [ ]:
# Share of floor pixels and median path loss per city
summ = maps.groupby(["split", "city"]).agg(floor_pct=("floor_frac", lambda v: 100 * v.mean()),
                                           median_pl_db=("pl_median_db", "median")).reset_index()
fig, ax = plt.subplots(figsize=(9, 3))
cols = summ.split.map({"train": "tab:gray", "val": "tab:green", "test": "tab:red"})
ax.bar(summ.city, summ.floor_pct, color=cols); ax.set_ylabel("floor pixels (%)"); ax.tick_params(axis="x", rotation=45)
fig.tight_layout(); fig.savefig("figures/floor_share_per_city.png", dpi=150); plt.show()
summ

## 5. Per-pixel model with selected features (LightGBM)

Choose any subset of the 17 channels in `FEATURES`. The per-pixel table is built from `PIX_PER_MAP` random outdoor pixels
of every training map, and the model is evaluated on `EVAL_PIX_PER_MAP` fixed pixels of each of the 480 test maps.

In [ ]:
FEATURE_SETS = {
    "all 17": CH,
    "geometry 7": ["height", "bmask", "tx_h", "logd2", "logd3", "elev", "los"],
    "link only": ["logd2", "logd3", "elev", "los", "ncross", "depth", "hmax_path", "h_exc", "nu"],
}
FEATURES = FEATURE_SETS["all 17"]

rng = np.random.default_rng(SEED)
train_ids = rng.permutation(tiles[tiles.split == "train"].scene_id)[:N_TRAIN_TILES]
val_ids = tiles[tiles.split == "val"].scene_id.tolist()
test_ids = tiles[tiles.split == "test"].scene_id.tolist()

def pixel_table(scene_ids, names, n_pix, seed):
    # Rows = sampled outdoor pixels of every map: X [n, len(names)], y [n], plus city / LoS / map index.
    X, Y, city, los, mid = [], [], [], [], []
    for s in scene_ids:
        t, c = load_tile(s), tiles.loc[tiles.scene_id == s, "city"].item()
        out = np.flatnonzero(t["outdoor"].ravel())
        for k in range(t["pathloss_db"].shape[0]):
            r = np.random.default_rng(seed + 1000 * k + int(hashlib.md5(s.encode()).hexdigest()[:6], 16))
            idx = r.choice(out, min(n_pix, len(out)), replace=False)
            f = feature_stack(t, k, names).reshape(len(names), -1)[:, idx].T
            X.append(f); Y.append(label(t["pathloss_db"][k]).ravel()[idx])
            city += [c] * len(idx); mid += [f"{s}#{k}"] * len(idx)
            los.append(feature_stack(t, k, ["los"]).ravel()[idx] > 0.5)
    return np.concatenate(X), np.concatenate(Y), np.array(city), np.concatenate(los), np.array(mid)

Xtr, ytr, *_ = pixel_table(train_ids, FEATURES, PIX_PER_MAP, SEED)
Xva, yva, *_ = pixel_table(val_ids[:5], FEATURES, 500, SEED + 1)
Xte, yte, cte, lte, mte = pixel_table(test_ids, FEATURES, EVAL_PIX_PER_MAP, 2027)
print(f"train {Xtr.shape}, val {Xva.shape}, test {Xte.shape}")

In [ ]:
import lightgbm as lgb, warnings
warnings.filterwarnings("ignore", message=".*eval_set.*")   # newer LightGBM versions prefer eval_X / eval_y

def rmse(e):
    return float(np.sqrt(np.mean(e ** 2)))

def report(name, pred, y=None, city=None, los=None):
    y = yte if y is None else y; city = cte if city is None else city; los = lte if los is None else los
    e, cov = pred - y, y < FLOOR_DB - 1e-3
    row = dict(model=name, rmse_all=rmse(e), rmse_covered=rmse(e[cov]), rmse_los=rmse(e[los]), rmse_nlos=rmse(e[~los]))
    row.update({f"rmse_{c}": rmse(e[city == c]) for c in sorted(set(city))})
    return row

lgbm = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.051, num_leaves=127, min_child_samples=50, subsample=0.6,
                         subsample_freq=1, colsample_bytree=0.6, reg_lambda=0.002, random_state=SEED, verbose=-1)
lgbm.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(100, verbose=False)])
pred_lgbm = lgbm.predict(Xte)
results = [report("LightGBM, " + ("all 17" if FEATURES == CH else f"{len(FEATURES)} features"), pred_lgbm)]
pd.DataFrame(results).round(2)

In [ ]:
imp = pd.Series(lgbm.booster_.feature_importance("gain"), index=FEATURES).sort_values()
ax = (imp / imp.sum()).plot.barh(figsize=(5, 4), color="tab:orange"); ax.set_xlabel("share of total gain")
plt.tight_layout(); plt.savefig("figures/lgbm_feature_importance.png", dpi=150); plt.show()

In [ ]:
# Compare feature subsets (same data, same settings)
for name, feats in FEATURE_SETS.items():
    if feats == FEATURES:
        continue
    cols = [FEATURES.index(f) for f in feats]
    m = lgb.LGBMRegressor(**lgbm.get_params()).fit(Xtr[:, cols], ytr, eval_set=[(Xva[:, cols], yva)],
                                                   callbacks=[lgb.early_stopping(100, verbose=False)])
    results.append(report(f"LightGBM, {name}", m.predict(Xte[:, cols])))
pd.DataFrame(results).round(2)

## 6. Spatial model with the same features (small U-Net)

The U-Net reads the selected channels as an image and predicts the whole map, so it can use neighbouring pixels.
It is trained with the mean-squared error over outdoor pixels and evaluated on the same test pixels as LightGBM.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as Fnn
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

def map_list(scene_ids):
    return [(s, k) for s in scene_ids for k in range(16)]

def norm_stats(scene_ids, names):
    xs, ys = [], []
    for s in scene_ids[:10]:
        t = load_tile(s)
        for k in range(0, 16, 4):
            xs.append(feature_stack(t, k, names).reshape(len(names), -1)[:, ::37])
            ys.append(label(t["pathloss_db"][k])[t["outdoor"]][::37])
    X, Y = np.concatenate(xs, 1), np.concatenate(ys)
    sd = X.std(1); sd[sd < 1e-6] = 1
    return X.mean(1).astype(np.float32), sd.astype(np.float32), float(Y.mean()), float(Y.std())

MU, SD, YMU, YSD = norm_stats(list(train_ids), FEATURES)

class Maps(torch.utils.data.Dataset):
    def __init__(self, scene_ids, names):
        self.items, self.names, self.cache = map_list(scene_ids), names, {}
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        s, k = self.items[i]
        if s not in self.cache:                       # tiles stay in memory after the first epoch
            self.cache[s] = load_tile(s)
        t = self.cache[s]
        x = (feature_stack(t, k, self.names) - MU[:, None, None]) / SD[:, None, None]
        y = (label(t["pathloss_db"][k]) - YMU) / YSD
        return torch.from_numpy(x), torch.from_numpy(y[None].astype(np.float32)), torch.from_numpy(t["outdoor"][None])

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.GELU(),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.GELU())

class UNet(nn.Module):
    def __init__(self, cin, base=12, levels=4):
        super().__init__()
        ch = [base * 2 ** i for i in range(levels + 1)]
        self.down = nn.ModuleList([block(cin, ch[0])] + [block(ch[i], ch[i + 1]) for i in range(levels)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(levels))])
        self.dec = nn.ModuleList([block(2 * ch[i], ch[i]) for i in reversed(range(levels))])
        self.head = nn.Conv2d(ch[0], 1, 1)
    def forward(self, x):
        skips = []
        for i, b in enumerate(self.down):
            x = b(x if i == 0 else Fnn.max_pool2d(x, 2))
            skips.append(x)
        x = skips.pop()
        for up, dec in zip(self.up, self.dec):
            x = dec(torch.cat([up(x), skips.pop()], 1))
        return self.head(x)

net = UNet(len(FEATURES)).to(dev)
print(f"U-Net parameters: {sum(p.numel() for p in net.parameters()) / 1e6:.2f} M on {dev}")

In [ ]:
dl = torch.utils.data.DataLoader(Maps(list(train_ids), FEATURES), batch_size=8, shuffle=True)
opt = torch.optim.AdamW(net.parameters(), lr=7.3e-4, weight_decay=2.4e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=7.3e-4, total_steps=UNET_EPOCHS * len(dl))
for ep in range(UNET_EPOCHS):
    net.train(); tot = 0.0
    for x, y, v in dl:
        x, y, v = x.to(dev), y.to(dev), v.to(dev)
        loss = (((net(x) - y) ** 2) * v).sum() / v.sum()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item()
    print(f"epoch {ep + 1}/{UNET_EPOCHS}: training RMSE {np.sqrt(tot / len(dl)) * YSD:.2f} dB")

In [ ]:
@torch.no_grad()
def predict_map(t, k):
    x = (feature_stack(t, k, FEATURES) - MU[:, None, None]) / SD[:, None, None]
    net.eval()
    return net(torch.from_numpy(x)[None].to(dev)).cpu().numpy()[0, 0] * YSD + YMU

# evaluate on exactly the same test pixels as LightGBM (pixel_table uses fixed seeds)
pred_unet, i0 = np.empty_like(yte), 0
for s in test_ids:
    t = load_tile(s); out = np.flatnonzero(t["outdoor"].ravel())
    for k in range(16):
        r = np.random.default_rng(2027 + 1000 * k + int(hashlib.md5(s.encode()).hexdigest()[:6], 16))
        idx = r.choice(out, min(EVAL_PIX_PER_MAP, len(out)), replace=False)
        pred_unet[i0:i0 + len(idx)] = predict_map(t, k).ravel()[idx]; i0 += len(idx)
results.append(report("U-Net, all 17" if FEATURES == CH else f"U-Net, {len(FEATURES)} features", pred_unet))
pd.DataFrame(results).round(2)

## 7. Predicted maps and errors

In [ ]:
s, k = test_ids[0], int(np.argmax(los_frac))
t = load_tile(s)
truth = np.where(t["outdoor"], label(t["pathloss_db"][k]), np.nan)
X = feature_stack(t, k, FEATURES).reshape(len(FEATURES), -1).T
p_l = np.where(t["outdoor"], lgbm.predict(X).reshape(t["outdoor"].shape), np.nan)
p_u = np.where(t["outdoor"], predict_map(t, k), np.nan)
fig, ax = plt.subplots(1, 5, figsize=(19, 3.6))
for a, img, ttl, cm, lim in [(ax[0], truth, "ray tracing", "viridis_r", (70, 160)), (ax[1], p_l, "LightGBM", "viridis_r", (70, 160)),
                             (ax[2], p_u, "U-Net", "viridis_r", (70, 160)), (ax[3], p_l - truth, "LightGBM error", "RdBu_r", (-25, 25)),
                             (ax[4], p_u - truth, "U-Net error", "RdBu_r", (-25, 25))]:
    im = a.imshow(img, origin="lower", extent=ext, cmap=cm, vmin=lim[0], vmax=lim[1]); a.set_title(ttl)
    a.plot(*t["tx_xyz_m"][k, :2], "r*", ms=10); fig.colorbar(im, ax=a, fraction=0.046, label="dB")
fig.tight_layout(); fig.savefig("figures/prediction_example.png", dpi=150); plt.show()

In [ ]:
R = pd.DataFrame(results)
city_cols = [c for c in R.columns if c.startswith("rmse_") and c not in ("rmse_all", "rmse_covered", "rmse_los", "rmse_nlos")]
ax = R.set_index("model")[["rmse_all", "rmse_covered", "rmse_los", "rmse_nlos"] + city_cols].T.plot.bar(figsize=(10, 3.5))
ax.set_ylabel("test RMSE (dB)"); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("figures/rmse_comparison.png", dpi=150); plt.show()
R.round(2)

## 8. Going further

* `results/` contains every result of the paper (per run and per test map), and the notebook `ML_vs_DL_radio_maps.ipynb`
  in the repository reproduces all tables and figures.
* `scenes/` holds the Sionna scenes, so the ray tracing can be repeated at other frequencies, materials or receiver heights.
* `checks/` holds LoS-only runs and a second Monte-Carlo seed for ten tiles (label noise σ ≈ 2.1 dB).

If you use UrbanRM7, please cite the paper (see `CITATION.cff` in the repository). Building data © OpenStreetMap contributors.